In [1]:
from pathlib import Path
import pandas as pd

In [2]:
CWD = Path.cwd()
DATA_FOLDER = CWD / "data"
OUTPUT_FOLDER = CWD / "data" / "processed"

First let's check the format of the data

In [3]:
with open(DATA_FOLDER / 'combined_data_1.txt', 'r') as file:
    first_lines = file.readlines()[:555]

for (line_index, line) in enumerate(first_lines):
    if (line_index < 10 or line_index > 545):
        print(f"Line {line_index}.\n{line.strip()}")

Line 0.
1:
Line 1.
1488844,3,2005-09-06
Line 2.
822109,5,2005-05-13
Line 3.
885013,4,2005-10-19
Line 4.
30878,4,2005-12-26
Line 5.
823519,3,2004-05-03
Line 6.
893988,3,2005-11-17
Line 7.
124105,4,2004-08-05
Line 8.
1248029,3,2004-04-22
Line 9.
1842128,4,2004-05-09
Line 546.
1426604,4,2005-09-01
Line 547.
1815755,5,2004-07-20
Line 548.
2:
Line 549.
2059652,4,2005-09-05
Line 550.
1666394,3,2005-04-19
Line 551.
1759415,4,2005-04-22
Line 552.
1959936,5,2005-11-21
Line 553.
998862,4,2004-11-13
Line 554.
2625420,2,2004-12-06


Now let's parse it to the proper dataframe

In [4]:
def parse_movie_ratings_file(filename):
    data = []
    current_movie_id = None

    with open(DATA_FOLDER / filename, 'r') as file:
        for (line_num, line) in enumerate(file):
            line = line.strip()

            # if it is a movie number
            if ':' in line:
                current_movie_id = int(line[:-1])

            # if it is a row, but we are cautious
            elif ',' in line:
                try:
                    parts = line.split(',')
                    if len(parts) == 3:
                        customer_id, rating, _ = parts
                        data.append({
                            'user_id': int(customer_id),
                            'movie_id': current_movie_id,
                            'rating': float(rating),
                        })
                    else: # if the structure is maybe different
                        print(f'Error parsing line {line_num}, which is: "{line}".')
                        raise Exception()
                except:
                    print(f'Error parsing line {line_num}, which is: "{line}".')

            else: # there shouldn't be any other row
                print(f'Error parsing line {line_num}, which is: "{line}".')
                raise Exception()

    return pd.DataFrame(data)

In [5]:
df1 = parse_movie_ratings_file('combined_data_1.txt')

Let's check if it is as desired

In [6]:
df1.head(10)

,user_id,movie_id,rating
0,1488844,1,3.0
1,822109,1,5.0
2,885013,1,4.0
3,30878,1,4.0
4,823519,1,3.0
5,893988,1,3.0
6,124105,1,4.0
7,1248029,1,3.0
8,1842128,1,4.0
9,2238063,1,3.0


Let's create the full dataframe

In [7]:
df2 = parse_movie_ratings_file('combined_data_2.txt')

In [8]:
df3 = parse_movie_ratings_file('combined_data_3.txt')

In [9]:
df4 = parse_movie_ratings_file('combined_data_4.txt')

In [10]:
df = pd.concat([df1, df2, df3, df4])

In [11]:
df

,user_id,movie_id,rating
0,1488844,1,3.0
1,822109,1,5.0
2,885013,1,4.0
3,30878,1,4.0
4,823519,1,3.0
...,...,...,...
26847518,1790158,17770,4.0
26847519,1608708,17770,3.0
26847520,234275,17770,1.0
26847521,255278,17770,4.0


Now let's split it into training and testing set.

In [12]:
df_shuffled = df.sample(frac=1, random_state=42).reset_index(drop=True)

split_idx = int(len(df_shuffled) * 0.9)

df_train = df_shuffled.iloc[:split_idx].copy()
df_test = df_shuffled.iloc[split_idx:].copy()

In [13]:
train_movies = set(df_train['movie_id'].unique())
df_test_filtered = df_test[df_test['movie_id'].isin(train_movies)].copy()

In [14]:
print(f"Dimensions of df: {df.shape}. Train df: {df_train.shape}. Test df: {df_test_filtered.shape}")

Dimensions of df: (100480507, 3). Train df: (90432456, 3). Test df: (10048051, 3)


In [15]:
train_users = set(df_train['user_id'].unique())
df_test_filtered_users = df_test_filtered[df_test_filtered['user_id'].isin(train_users)].copy()

In [16]:
print(f"Train df: {df_train.shape}. Test df: {df_test_filtered_users.shape}")

Train df: (90432456, 3). Test df: (10047853, 3)


In [17]:
df_train.head(10)

,user_id,movie_id,rating
0,404578,15582,4.0
1,636396,6510,4.0
2,2258880,15500,4.0
3,1574865,14621,2.0
4,237063,4123,4.0
5,1904519,14454,5.0
6,2592898,10879,4.0
7,22096,14364,1.0
8,1426829,12195,4.0
9,914155,3638,5.0


In [18]:
df_test_filtered_users.head(10)

,user_id,movie_id,rating
90432456,892176,6692,3.0
90432457,1832243,8204,4.0
90432458,1449380,10854,5.0
90432459,1829661,9074,3.0
90432460,2628417,16002,3.0
90432461,775615,15307,4.0
90432462,2609907,6347,3.0
90432463,391521,15164,3.0
90432464,174107,6508,4.0
90432465,2564453,3925,3.0


In [19]:
df_train.to_csv(OUTPUT_FOLDER / 'training_data.csv', index=False)
df_test_filtered_users.to_csv(OUTPUT_FOLDER / 'testing_data.csv', index=False)